# Using Ragas to Evaluate a RAG Application

## Dependencies and API Keys:

> NOTE: Please skip the pip install commands if you are running the notebook locally.

In [1]:
!pip install -q ragas datasets faiss-cpu sentence_transformers langchain_openai langchain_core langchain langchain_community langchain_huggingface

We'll also need to provide our API keys.

First, OpenAI's for our LLM/embedding model combination!

In [2]:
import os
from getpass import getpass
os.environ["OPENAI_API_KEY"] = getpass("Please enter your OpenAI API key!")

OPTIONALLY:

We can also provide a Ragas API key - which you can sign-up for [here](https://app.ragas.io/).

In [3]:
os.environ["RAGAS_APP_TOKEN"] = getpass("Please enter your Ragas API key!")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

In [4]:
!mkdir data

mkdir: data: File exists


In [6]:
!curl https://simonwillison.net/2023/Dec/31/ai-in-2023/ -o data/2023_llms.html

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 31392    0 31392    0     0  66128      0 --:--:-- --:--:-- --:--:-- 66227


In [7]:
!curl https://simonwillison.net/2024/Dec/31/llms-in-2024/ -o data/2024_llms.html

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 70251    0 70251    0     0   350k      0 --:--:-- --:--:-- --:--:--  351k


Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [8]:
from langchain_community.document_loaders import DirectoryLoader
from pathlib import Path

loader = DirectoryLoader(
    'data',
    glob="*.html",
    show_progress=False
)
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [9]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

generating personas, scenarios, samples

In [10]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/2 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/2 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/12 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/26 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [11]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,Wht is EleutherAI?,[Code may be the best application The ethics o...,EleutherAI is one of the organizations that ha...,single_hop_specifc_query_synthesizer
1,What significant event related to AI research ...,[Based Development As a computer scientist and...,"In September last year, the term 'prompt injec...",single_hop_specifc_query_synthesizer
2,What AI mean?,[Simon Willison’s Weblog Subscribe Stuff we fi...,"AI refers to Artificial Intelligence, which in...",single_hop_specifc_query_synthesizer
3,Who is Simon Willison and what did he write ab...,[easy to follow. The rest of the document incl...,Simon Willison is a writer who posted content ...,single_hop_specifc_query_synthesizer
4,How do the challenges of understanding and con...,[<1-hop>\n\nCode may be the best application T...,The challenges of understanding and controllin...,multi_hop_abstract_query_synthesizer
5,How do Large Language Models (LLMs) function a...,[<1-hop>\n\nCode may be the best application T...,Large Language Models (LLMs) function as black...,multi_hop_abstract_query_synthesizer
6,How do the challenges of understanding and con...,[<1-hop>\n\nCode may be the best application T...,The challenges of understanding and controllin...,multi_hop_abstract_query_synthesizer
7,How does the ethics of training data impact th...,[<1-hop>\n\nCode may be the best application T...,The ethics of training data significantly impa...,multi_hop_abstract_query_synthesizer
8,How does the use of Claude in building tools a...,"[<1-hop>\n\nhere, but getting to that value is...",The use of Claude in building tools and applic...,multi_hop_specific_query_synthesizer
9,What were the key advancements in Large Langua...,[<1-hop>\n\nSimon Willison’s Weblog Subscribe ...,"In 2024, significant advancements in Large Lan...",multi_hop_specific_query_synthesizer


#### OPTIONAL:

If you've provided your Ragas API key - you can use this web interface to look at the created data!

In [12]:
dataset.upload()

Testset uploaded! View at https://app.ragas.io/dashboard/alignment/testset/238ac594-9c9d-4195-8aaf-c72402dceb57


'https://app.ragas.io/dashboard/alignment/testset/238ac594-9c9d-4195-8aaf-c72402dceb57'

## LangChain RAG

Now we'll construct our LangChain RAG, which we will be evaluating using the above created test data!

### R - Retrieval

Let's start with building our retrieval pipeline, which will involve loading the same data we used to create our synthetic test set above.

> NOTE: We need to use the same data - as our test set is specifically designed for this data.

In [13]:
path = "data/"
loader = DirectoryLoader(path, glob="*.html")
docs = loader.load()

Now that we have our data loaded, let's split it into chunks!

In [14]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
split_documents = text_splitter.split_documents(docs)
len(split_documents)

74

In [15]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

Now we can build our in memory QDrant vector store.

In [16]:
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

client = QdrantClient(":memory:")

client.create_collection(
    collection_name="llms",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="llms",
    embedding=embeddings,
)

We can now add our documents to our vector store.

In [17]:
_ = vector_store.add_documents(documents=split_documents)

Let's define our retriever.

In [18]:
retriever = vector_store.as_retriever(search_kwargs={"k": 5})

Now we can produce a node for retrieval!

In [19]:
def retrieve(state):
  retrieved_docs = retriever.invoke(state["question"])
  return {"context" : retrieved_docs}

### Augmented

Let's create a simple RAG prompt!

In [20]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
You are a helpful assistant who answers questions based on provided context. You must only use the provided context, and cannot use your own knowledge.

### Question
{question}

### Context
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

### Generation

We'll also need an LLM to generate responses - we'll use `gpt-4o-mini` to avoid using the same model as our judge model.

In [21]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

Then we can create a `generate` node!

In [22]:
def generate(state):
  docs_content = "\n\n".join(doc.page_content for doc in state["context"])
  messages = rag_prompt.format_messages(question=state["question"], context=docs_content)
  response = llm.invoke(messages)
  return {"response" : response.content}

### Building RAG Graph with LangGraph

Let's create some state for our LangGraph RAG graph!

In [23]:
from langgraph.graph import START, StateGraph
from typing_extensions import List, TypedDict
from langchain_core.documents import Document

class State(TypedDict):
  question: str
  context: List[Document]
  response: str

Now we can build our simple graph!

> NOTE: We're using `add_sequence` since we will always move from retrieval to generation. This is essentially building a chain in LangGraph.

In [24]:
graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
graph = graph_builder.compile()

Let's do a test to make sure it's doing what we'd expect.

In [25]:
response = graph.invoke({"question" : "How are LLM agents useful?"})

In [26]:
response["response"]

'LLM agents are useful in several ways as highlighted in the context:\n\n1. **Ease of Building**: LLMs can be built with relatively simple code, requiring only a few hundred lines of Python, which makes them accessible to more people than just the super-rich.\n\n2. **Running on Personal Devices**: Advancements in LLM technology have made it possible to run useful models on personal computers, broadening accessibility and usability.\n\n3. **Code Generation**: LLMs are particularly effective at writing code. They can generate code, execute it, and revise it based on feedback (like error messages), which mitigates the issues of inaccuracies or "hallucinations" in coding.\n\n4. **Agent Functionality**: While many prototypes exist, the concept of AI agents that can operate independently on behalf of users is exciting and represents potential future applications of LLMs.\n\n5. **Addressing Criticism**: There is a recognition of the need for critique and discussion regarding the ethical and e

## Evaluating the App with Ragas

Now we can finally do our evaluation!

We'll start by running the queries we generated usign SDG above through our application to get context and responses.

In [27]:
for test_row in dataset:
  response = graph.invoke({"question" : test_row.eval_sample.user_input})
  test_row.eval_sample.response = response["response"]
  test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]

In [28]:
dataset.to_pandas()

,user_input,retrieved_contexts,reference_contexts,response,reference,synthesizer_name
0,Wht is EleutherAI?,"[If you can gather the right data, and afford ...",[Code may be the best application The ethics o...,EleutherAI is mentioned as one of the organiza...,EleutherAI is one of the organizations that ha...,single_hop_specifc_query_synthesizer
1,What significant event related to AI research ...,[OpenAI are not the only game in town here. Go...,[Based Development As a computer scientist and...,The provided context does not mention any sign...,"In September last year, the term 'prompt injec...",single_hop_specifc_query_synthesizer
2,What AI mean?,[A lot of people are excited about AI agents—a...,[Simon Willison’s Weblog Subscribe Stuff we fi...,"AI stands for ""artificial intelligence."" In th...","AI refers to Artificial Intelligence, which in...",single_hop_specifc_query_synthesizer
3,Who is Simon Willison and what did he write ab...,[Simon Willison’s Weblog\n\nSubscribe\n\nStuff...,[easy to follow. The rest of the document incl...,Simon Willison is a writer and blogger who foc...,Simon Willison is a writer who posted content ...,single_hop_specifc_query_synthesizer
4,How do the challenges of understanding and con...,[Another common technique is to use larger mod...,[<1-hop>\n\nCode may be the best application T...,The challenges associated with understanding a...,The challenges of understanding and controllin...,multi_hop_abstract_query_synthesizer
5,How do Large Language Models (LLMs) function a...,[A lot of people are yet to be sold on their v...,[<1-hop>\n\nCode may be the best application T...,Large Language Models (LLMs) function as black...,Large Language Models (LLMs) function as black...,multi_hop_abstract_query_synthesizer
6,How do the challenges of understanding and con...,[Another common technique is to use larger mod...,[<1-hop>\n\nCode may be the best application T...,The challenges of understanding and controllin...,The challenges of understanding and controllin...,multi_hop_abstract_query_synthesizer
7,How does the ethics of training data impact th...,"[Since then, almost every major LLM (and most ...",[<1-hop>\n\nCode may be the best application T...,The ethics of training data significantly impa...,The ethics of training data significantly impa...,multi_hop_abstract_query_synthesizer
8,How does the use of Claude in building tools a...,"[If anything, this problem got worse in 2024.\...","[<1-hop>\n\nhere, but getting to that value is...",The use of Claude in building tools and applic...,The use of Claude in building tools and applic...,multi_hop_specific_query_synthesizer
9,What were the key advancements in Large Langua...,[The rise of inference-scaling “reasoning” mod...,[<1-hop>\n\nSimon Willison’s Weblog Subscribe ...,"In 2024, key advancements in Large Language Mo...","In 2024, significant advancements in Large Lan...",multi_hop_specific_query_synthesizer


Then we can convert that table into a `EvaluationDataset` which will make the process of evaluation smoother.

In [29]:
from ragas import EvaluationDataset

evaluation_dataset = EvaluationDataset.from_pandas(dataset.to_pandas())

We'll need to select a judge model - in this case we're using the same model that was used to generate our Synthetic Data.

In [30]:
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o"))

Next up - we simply evaluate on our desired metrics!

In [31]:
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
from ragas import evaluate, RunConfig

custom_run_config = RunConfig(timeout=360)

result = evaluate(
    dataset=evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
    llm=evaluator_llm,
    run_config=custom_run_config
)
result

Evaluating:   0%|          | 0/72 [00:00<?, ?it/s]

Exception raised in Job[24]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o in organization org-NfQAqRGP9ng2xxzqwEc3LBD2 on tokens per min (TPM): Limit 30000, Used 28817, Requested 2499. Please try again in 2.632s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}})
Exception raised in Job[28]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o in organization org-NfQAqRGP9ng2xxzqwEc3LBD2 on tokens per min (TPM): Limit 30000, Used 29653, Requested 1873. Please try again in 3.052s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}})
Exception raised in Job[1]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o in organization org-NfQAqRGP9ng2xxzqwEc3LBD2 on tokens per min (TPM): Limit 30000, Used 29306, Requested

{'context_recall': 0.6429, 'faithfulness': 0.9905, 'factual_correctness': 0.4873, 'answer_relevancy': 0.7155, 'context_entity_recall': 0.3270, 'noise_sensitivity_relevant': 0.3369}

In [33]:
import time

# Initialize fine-tuned embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

finetuned_embeddings = HuggingFaceEmbeddings(
    model_name="lsy9874205/legal-ft-2"  # Or path to your fine-tuned model
)

# Create new vector store for fine-tuned model
finetuned_client = QdrantClient(":memory:")
finetuned_client.create_collection(
    collection_name="ai_across_years_finetuned",
    vectors_config=VectorParams(size=1024, distance=Distance.COSINE),
)

finetuned_vector_store = QdrantVectorStore(
    client=finetuned_client,
    collection_name="ai_across_years_finetuned",
    embedding=finetuned_embeddings,
)

# Add the same documents
_ = finetuned_vector_store.add_documents(documents=split_documents)

# Create retriever with fine-tuned embeddings
finetuned_retriever = finetuned_vector_store.as_retriever(search_kwargs={"k": 5})

# Create new graph with fine-tuned retriever
def retrieve_finetuned(state):
    retrieved_docs = finetuned_retriever.invoke(state["question"])
    return {"context": retrieved_docs}

graph_builder_finetuned = StateGraph(State).add_sequence([retrieve_finetuned, generate])
graph_builder_finetuned.add_edge(START, "retrieve_finetuned")
graph_finetuned = graph_builder_finetuned.compile()

# Run evaluation with fine-tuned model
for test_row in dataset:
    response = graph_finetuned.invoke({"question": test_row.eval_sample.user_input})
    test_row.eval_sample.response = response["response"]
    test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]
    time.sleep(2)  # To avoid rate limiting

# Evaluate fine-tuned model
result_finetuned = evaluate(
    dataset=evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
    llm=evaluator_llm,
    run_config=custom_run_config
)

print("\nBase Model Results:")
print(result)
print("\nFine-tuned Model Results:")
print(result_finetuned)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/275 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/29.0k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/641 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

1_Pooling%2Fconfig.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

Evaluating:   0%|          | 0/72 [00:00<?, ?it/s]

Exception raised in Job[7]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o in organization org-NfQAqRGP9ng2xxzqwEc3LBD2 on tokens per min (TPM): Limit 30000, Used 29892, Requested 2167. Please try again in 4.118s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}})
Exception raised in Job[25]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o in organization org-NfQAqRGP9ng2xxzqwEc3LBD2 on tokens per min (TPM): Limit 30000, Used 29702, Requested 1936. Please try again in 3.276s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}})
Exception raised in Job[13]: RateLimitError(Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o in organization org-NfQAqRGP9ng2xxzqwEc3LBD2 on tokens per min (TPM): Limit 30000, Used 29528, Requested


Base Model Results:
{'context_recall': 0.6429, 'faithfulness': 0.9905, 'factual_correctness': 0.4873, 'answer_relevancy': 0.7155, 'context_entity_recall': 0.3270, 'noise_sensitivity_relevant': 0.3369}

Fine-tuned Model Results:
{'context_recall': 0.6143, 'faithfulness': 0.8696, 'factual_correctness': 0.4556, 'answer_relevancy': 0.7155, 'context_entity_recall': 0.4283, 'noise_sensitivity_relevant': 0.3644}
